middleware

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


summarisation middleware

In [27]:

### summarisation  middleware 

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

### message based summarisation

agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    # model_provider="groq",  # Set your provider here
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            model_provider="groq",
            trigger=("messages",10),
            keep=("messages",4))
    ]
)

In [31]:
### run with thread

config={"configurable":{"thread_id":"test-1"}}

In [33]:
# alternatives test data

questions=[
    "what is 2+2?", 
    "what is 2+6?", 
    "what is 2*8?", 
    "what is 10/5?",
    "what is 10*5?",
    "what is 10+10?"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response["messages"])}")



Messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='f1755627-fd3d-4d44-a7fa-3bce7c714c27'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "what is 2+2?"\n2.  **Identify Core Question:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy:** 2+2 is universally 4 in standard base-10 arithmetic.\n6.  **Output Generation:** "2 + 2 equals 4." (or similar)✅\n</think>\n\n2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 17, 'total_tokens': 155, 'completion_time': 0.263839651, 'completion_tokens_details': None, 'prompt_time': 0.001010123, 'prompt_tokens_details': None, 'queue_time': 0.042805463, 'total_time': 0.264849774}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d'

Token size

In [36]:
from numpy import character
from langchain import tools

### summarisation  middleware 

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool


@tool
def search_hotels(city:str)->str:
    """search hotels -return long response to use more tokens"""
    return f"""Hotels in {city} :
    1. Grand Hotel - 5 Star , $350/night, spa, pool,gym
    2. City Inn - 4 Star , $250/night, spa, pool
    3. Budget Stay - 3 Star , $150/night, spa
    """

agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            model_provider="groq",
            trigger=("tokens",550),
            keep=("tokens",200))
    ]
)

config={"configurable":{"thread_id":"test-1"}}

#token counter (approximation)

def count_tokens(messages):
    total_char=sum(len(str(m.content))for m in messages)
    return total_char //4 #4 char = 1 token





    


In [ ]:
#run test

cities=["Paris","New York","London","Tokyo","Dubai","Singapore","Mumbai"]

for city in cities:
    response=agent.invoke({
        "messages":[
            HumanMessage(content=f"Find Hotels in {city}")
        ]
    },config=config)

    tokens=count_tokens(response["messages"])
    print(f"{city}: ~ {tokens},{len(response['messages'])}messages")
    print(f"{(response['messages'])}")

Paris: ~ tokens,4messages
[HumanMessage(content='Find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='4e10ad60-c325-45df-b27c-3b33a4b242c0'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify the user\'s intent**: The user wants to find hotels in Paris.\n2.  **Check available tools**: I have access to a `search_hotels` function.\n3.  **Analyze tool parameters**: The `search_hotels` function requires a `city` parameter (string).\n4.  **Extract parameters**: The city is "Paris".\n5.  **Construct tool call**: Call `search_hotels` with `city="Paris"`.\n6.  **Execute tool call**.\n7.  **Process response**: Present the results to the user.\n\nLet\'s call the function.\n', 'tool_calls': [{'id': 'c1y505qmz', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 277, 'total_tokens': 445, 'completion_time': 0.32877